# NOAA-20 detection forecast for every trained cell in central Spain

Open this notebook in Colab and run the cells in order. The default request issues a forecast at **2026-11-15 00:00 UTC** for all **288 model cells**. It scores **15–17 November UTC** using the three saved CatBoost models. The 10-day request is recorded in the output, and the date axis contains only dates supported by the trained horizons. A missing score within these horizons is represented by JSON `null`.

- `forecast_grid(date, days=10)` returns dates, grid cells and a probability matrix indexed by **day, then cell**.
- `predict(date, latitude, longitude, days=10)` returns the same daily series for the trained cell containing the requested coordinates. Bounds are west/south inclusive and east/north exclusive.
- `spain_forecast.json` stores the daily forecast for all trained cells. If `spain.json` is available, its historical data is copied and the daily output is added under `forecast`. The source file is kept intact.

The probabilities describe NOAA-20 FIRMS detections per 0.25° cell and UTC day. Earth Engine supplies current and trailing weather and moisture inputs. Static terrain and land cover and CatBoost files come from Drive. FIRMS detections are used in training labels only. The example date can be changed in the configuration cell.


In [ ]:
from collections import Counter
from functools import lru_cache
from hashlib import sha256
from importlib.util import find_spec
from pathlib import Path
from time import sleep
import json
import subprocess
import sys

from google.colab import drive, userdata
drive.mount("/content/drive")

DRIVE_ROOT = Path("/content/drive/MyDrive/spain_firms_2012_2025")
ISSUE_DATE = "2026-11-15"  # Date of issue at 00:00 UTC.
REQUESTED_DAYS = 10
RUN_DIR = None            # None selects the latest complete NOAA-20 model export on Drive.
EE_PROJECT = ""           # Or use the EE_PROJECT Colab secret.
HISTORY_JSON = None       # Or set Path(...) to the supplied spain.json to append the forecast.
OUTPUT_PATH = None        # Or set Path(...) to an explicit export path on Drive.


def install_if_missing(packages):
    missing = [package for module, package in packages.items() if find_spec(module) is None]
    if missing:
        subprocess.check_call([sys.executable, "-m", "pip", "install", "-q", *missing])


install_if_missing({"catboost": "catboost>=1.2.8,<2", "pyarrow": "pyarrow"})
install_if_missing({"ee": "earthengine-api", "xclim": "xclim==0.59.1"})

import numpy as np
import pandas as pd
from catboost import CatBoostClassifier
from IPython.display import display

RUN_ROOT = DRIVE_ROOT / "model_runs" / "noaa20_environment_catboost_3day"
HORIZONS = (1, 2, 3)
MODEL_FILES = {day: f"catboost_day{day}.cbm" for day in HORIZONS}
manifest_name = "model_manifest.json"
if RUN_DIR is None:
    complete_runs = [p for p in RUN_ROOT.glob("*") if (p / manifest_name).is_file()
                     and all((p / name).is_file() for name in MODEL_FILES.values())]
    if not complete_runs:
        raise FileNotFoundError(f"No complete CatBoost export in {RUN_ROOT}. Run the training notebook's save cell or set RUN_DIR.")
    RUN_DIR = max(complete_runs)
RUN_DIR = Path(RUN_DIR)
manifest = json.loads((RUN_DIR / manifest_name).read_text(encoding="utf-8"))
if manifest["target"].get("platform") != "NOAA20" or manifest.get("firms_usage") != "labels_only":
    raise ValueError("Select the NOAA-20 model export with FIRMS used only for labels.")
if manifest["forecast_window_start_offsets_days"] != {str(day): day - 1 for day in HORIZONS}:
    raise ValueError("The model export uses unexpected forecast windows.")

FEATURES = {day: manifest["features_by_lead"][str(day)] for day in HORIZONS}
all_features = list(dict.fromkeys(name for day in HORIZONS for name in FEATURES[day]))
models = {day: CatBoostClassifier().load_model(str(RUN_DIR / name)) for day, name in MODEL_FILES.items()}
for day, model in models.items():
    if model.feature_names_ != FEATURES[day]:
        raise ValueError(f"Day {day} model features disagree with the manifest. Use files from one training run.")

WEST, SOUTH, EAST, NORTH = manifest["bbox_wsen"]
CELL_DEG = manifest["cell_degrees"]
ys = np.arange(round(SOUTH / CELL_DEG), round(NORTH / CELL_DEG))
xs = np.arange(round(WEST / CELL_DEG), round(EAST / CELL_DEG))
CELL_KEYS = ["cell_y", "cell_x"]
KEYS = ["issue_utc", *CELL_KEYS]
CELL_INDEX = pd.MultiIndex.from_product([ys, xs], names=CELL_KEYS)
grid_cells = CELL_INDEX.to_frame(index=False)
RUN_AGE_HOURS = 6
GFS_DATASET = "NOAA/GFS0P25"
GFS_BASE = ["temp_max_24h", "rh_min_24h", "wind_max_24h", "precip_sum_24h"]
GFS = {day: [f"gfs_day{day}_{name}" for name in GFS_BASE] for day in HORIZONS}
TERRAIN = ["elevation_mean_m", "slope_mean_deg"]
LAND = ["forest_frac", "shrubland_frac", "natural_grassland_frac", "agriculture_frac"]
SMAP = ["smap_surface_daily", "smap_surface_mean_7d"]
FEATURE_DIR = DRIVE_ROOT / "features"
OUTPUT_ROOT = DRIVE_ROOT / "forecast_runs" / "noaa20_environment_catboost_3day"
MISSING_PIXEL = -9999.0
transform = [CELL_DEG, 0, WEST, 0, -CELL_DEG, NORTH]
pixel_grid = {
    "dimensions": {"width": len(xs), "height": len(ys)},
    "affineTransform": dict(zip(["scaleX", "shearX", "translateX", "shearY", "scaleY", "translateY"], transform)),
    "crsCode": "EPSG:4326",
}


def utc(values):
    return pd.to_datetime(values, utc=True, errors="raise").astype("datetime64[ns, UTC]")


def secret(name):
    try:
        return (userdata.get(name) or "").strip()
    except (userdata.SecretNotFoundError, userdata.NotebookAccessError):
        return ""


def align_cells(frame):
    indexed = frame.set_index(CELL_KEYS)
    if not indexed.index.is_unique or not indexed.index.isin(CELL_INDEX).all():
        raise ValueError("Input contains duplicate or out-of-grid cell keys.")
    return indexed.reindex(CELL_INDEX)


def numeric(values):
    parsed = pd.to_numeric(values, errors="raise").astype(float)
    return parsed.where(np.isfinite(parsed) & parsed.ne(MISSING_PIXEL))


print(f"Model export: {RUN_DIR.name} · {len(CELL_INDEX)} grid cells")

def issue_timestamp(date):
    value = pd.to_datetime(date, utc=True, errors="raise")
    if pd.isna(value) or value != value.normalize():
        raise ValueError("Pass a UTC calendar date such as 2026-11-15.")
    if value > pd.Timestamp.now(tz="UTC"):
        raise ValueError("The requested 00:00 UTC issue time is still in the future.")
    return value.as_unit("ns")


print(f"Model grid: west {WEST}, south {SOUTH}, east {EAST}, north {NORTH}.")


## Earth Engine and GFS weather

The GFS cycle for each issue is the preceding day's **18:00 UTC** cycle. The requested issue uses Days 1–3 from that same cycle. Trailing predictors use Day 1 forecasts from their original issue dates. For forecasting on a different date, the notebook assembles the corresponding inputs again.


In [ ]:
import ee

project = EE_PROJECT.strip() or secret("EE_PROJECT")
if not project:
    raise ValueError("Set EE_PROJECT above or add it to Colab Secrets.")
try:
    ee.Initialize(project=project)
except ee.EEException:
    ee.Authenticate(auth_mode="notebook", force=True)
    ee.Initialize(project=project)


In [ ]:
GFS_STATE_BANDS = [
    "temperature_2m_above_ground", "relative_humidity_2m_above_ground",
    "u_component_of_wind_10m_above_ground", "v_component_of_wind_10m_above_ground",
]
GFS_RAIN_BAND = "total_precipitation_surface"


def weather_leads(day):
    first = RUN_AGE_HOURS + 24 * (day - 1)
    return tuple(range(first, first + 24, 3)), tuple(range(first + 6, first + 25, 6))


def gfs_day_image(archive, issue, day):
    run = issue - pd.Timedelta(hours=RUN_AGE_HOURS)
    cycle = archive.filter(ee.Filter.eq("creation_time", run.value // 10**6))
    state_leads, rain_leads = weather_leads(day)

    def step(lead, bands, names):
        valid = run + pd.Timedelta(hours=lead)
        matching = cycle.filter(ee.Filter.eq("forecast_hours", lead)).filter(
            ee.Filter.eq("forecast_time", valid.value // 10**6)
        )
        missing = ee.Image.constant([0] * len(names)).rename(names).selfMask().toFloat()
        image = ee.Image(matching.first()).select(bands, names).toFloat().resample("bilinear")
        image = image.reproject(crs="EPSG:4326", crsTransform=transform)
        return ee.Image(ee.Algorithms.If(matching.size().eq(1), image, missing))

    def state(lead):
        inputs = step(lead, GFS_STATE_BANDS, ["temperature", "humidity", "u", "v"])
        wind = inputs.select("u").hypot(inputs.select("v")).rename("wind")
        return inputs.select(["temperature", "humidity"]).addBands(wind)

    def reduce_complete(collection, field, reducer, count):
        band = collection.select(field)
        return getattr(band, reducer)().updateMask(band.count().eq(count))

    states = ee.ImageCollection.fromImages([state(lead) for lead in state_leads])
    rain = ee.ImageCollection.fromImages([step(lead, [GFS_RAIN_BAND], ["rain"]) for lead in rain_leads])
    images = [
        reduce_complete(states, "temperature", "max", len(state_leads)),
        reduce_complete(states, "humidity", "min", len(state_leads)),
        reduce_complete(states, "wind", "max", len(state_leads)),
        reduce_complete(rain, "rain", "sum", len(rain_leads)),
    ]
    return ee.Image.cat(*images).rename(GFS[day]).toFloat().unmask(MISSING_PIXEL)



def compute_pixels(image, label):
    request = {"expression": image, "fileFormat": "NUMPY_NDARRAY", "grid": pixel_grid}
    for attempt in range(3):
        try:
            pixels = ee.data.computePixels(request)
            break
        except ee.EEException as error:
            retryable = any(token in str(error).lower() for token in (
                "429", "500", "502", "503", "504", "deadline", "temporarily", "timed out",
            ))
            if attempt == 2 or not retryable:
                raise
            sleep(2 ** (attempt + 1))
    if pixels.shape != (len(ys), len(xs)):
        raise ValueError(f"Unexpected {label} grid shape: {pixels.shape}.")
    return pixels


def fetch_weather(days):
    archive = ee.ImageCollection(GFS_DATASET)
    issue = days[-1]
    run = issue - pd.Timedelta(hours=RUN_AGE_HOURS)
    cycle = archive.filter(ee.Filter.eq("creation_time", run.value // 10**6))
    leads = Counter(int(lead) for lead in cycle.aggregate_array("forecast_hours").getInfo())
    if not leads:
        raise RuntimeError(f"GFS cycle {run:%Y-%m-%d %H:%M} UTC is unavailable in Earth Engine.")
    for day in HORIZONS:
        missing = sorted(lead for lead in set().union(*weather_leads(day)) if leads[lead] != 1)
        if missing:
            print(f"Day {day} missing or duplicate GFS forecast hours: {missing}")

    tables = []
    for start in range(0, len(days), 4):
        block = days[start:start + 4]
        images, names = [], {}
        for date in block:
            leads_for_date = HORIZONS if date == issue else (1,)
            names[date] = [name for day in leads_for_date for name in GFS[day]]
            images.extend(
                gfs_day_image(archive, date, day).rename([f"d{date:%Y%m%d}_{name}" for name in GFS[day]])
                for day in leads_for_date
            )
        pixels = compute_pixels(ee.Image.cat(*images), "GFS")
        for date in block:
            table = grid_cells.assign(issue_utc=date)
            for name in names[date]:
                values = numeric(pd.Series(pixels[f"d{date:%Y%m%d}_{name}"][::-1].ravel()))
                table[name] = values.where(values.ge(-273.15)) if "temp_max" in name else values.clip(lower=0)
            tables.append(table)
    return pd.concat(tables, ignore_index=True)


## Terrain, land cover and moisture history

SMAP keeps the original eight-day offset from issue date and a complete seven-day trailing mean. Terra NDMI uses the original composite QA, minimum valid coverage and release buffer. Observation dates are retained for the models that were trained with observation-age features.


In [ ]:
def cell_regions(ee, cells):
    return ee.FeatureCollection([
        ee.Feature(
            ee.Geometry.Rectangle(
                [x * CELL_DEG, y * CELL_DEG, (x + 1) * CELL_DEG, (y + 1) * CELL_DEG],
                proj="EPSG:4326",
                geodesic=False,
            ),
            {"cell_y": int(y), "cell_x": int(x)},
        )
        for y, x in cells[CELL_KEYS].itertuples(index=False, name=None)
    ])


def fetch_smap_block(ee, smap, projection, regions, cells, days):
    # Keep the band present on archive days with no usable frames.
    masked = ee.Image.constant(0).rename("sm_surface").toFloat().selfMask()
    fallback = ee.ImageCollection.fromImages([masked])

    images = []
    band_dates = {}
    for day in days:
        start = day.strftime("%Y-%m-%d")
        end = (day + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        day_id = day.strftime("d%Y%m%d")
        frames = smap.filterDate(start, end).merge(fallback)
        daily = frames.mean().updateMask(frames.count().eq(8))
        images.append(daily.toFloat().set("system:index", day_id))
        band_dates[f"{day_id}_sm_surface"] = day

    stats = ee.ImageCollection.fromImages(images).toBands().reduceRegions(
        collection=regions,
        reducer=ee.Reducer.mean(),
        crs=projection["crs"],
        crsTransform=projection["transform"],
    )
    wide = ee.data.computeFeatures({"expression": stats, "fileFormat": "PANDAS_DATAFRAME"})
    expected_cells = pd.MultiIndex.from_frame(cells)
    returned_cells = pd.MultiIndex.from_frame(wide[CELL_KEYS])
    if not returned_cells.is_unique or not expected_cells.sort_values().equals(returned_cells.sort_values()):
        raise ValueError("SMAP grid keys differ from the forecast grid.")

    daily = wide.reindex(columns=[*CELL_KEYS, *band_dates]).melt(
        id_vars=CELL_KEYS, var_name="band", value_name="sm_surface",
    )
    daily[SMAP_SOURCE_DAY] = daily.pop("band").map(band_dates)
    daily["sm_surface"] = pd.to_numeric(daily["sm_surface"], errors="raise")
    return daily


def ndmi_image(image):
    state = image.select("StateQA")
    quality = image.select("QA")
    nir = image.select("sur_refl_b02")
    swir = image.select("sur_refl_b06")

    cloud_and_snow_flags = (1 << 2) | (3 << 8) | (1 << 10) | (1 << 12) | (1 << 13) | (1 << 15)
    band_quality_flags = (15 << 6) | (15 << 22)
    valid = (
        state.bitwiseAnd(3).eq(0)
        .And(state.rightShift(3).bitwiseAnd(7).eq(1))
        .And(state.bitwiseAnd(cloud_and_snow_flags).eq(0))
        .And(quality.bitwiseAnd(3).lt(2))
        .And(quality.bitwiseAnd(band_quality_flags).eq(0))
        .And(nir.gte(0)).And(nir.lte(16000))
        .And(swir.gte(0)).And(swir.lte(16000))
        .And(nir.add(swir).gt(0))
    )
    ndmi = (
        image.normalizedDifference(["sur_refl_b02", "sur_refl_b06"])
        .updateMask(valid)
        .rename("ndmi_mean")
    )
    coverage = ndmi.mask().unmask(0, sameFootprint=False).rename(NDMI_COVERAGE)
    return ndmi.addBands(coverage).toFloat()


def fetch_ndmi_block(ee, modis, projection, regions, cells, block):
    images = []
    bands_by_day = {}
    # Name the output bands directly, independently of MODIS asset IDs.
    for day in block:
        start = day.strftime("%Y-%m-%d")
        stop = (day + pd.Timedelta(days=1)).strftime("%Y-%m-%d")
        image = ee.Image(modis.filterDate(start, stop).first())
        prefix = day.strftime("d%Y%m%d_")
        names = [prefix + "ndmi_mean", prefix + NDMI_COVERAGE]
        bands_by_day[day] = names
        images.append(ndmi_image(image).rename(names))

    stats = ee.Image.cat(*images).reduceRegions(
        collection=regions, reducer=ee.Reducer.mean(),
        crs=projection["crs"], crsTransform=projection["transform"],
    )
    wide = ee.data.computeFeatures({"expression": stats, "fileFormat": "PANDAS_DATAFRAME"})
    expected = pd.MultiIndex.from_frame(cells).sort_values()
    received = pd.MultiIndex.from_frame(wide[CELL_KEYS]).sort_values()
    if not received.equals(expected):
        raise ValueError("Earth Engine returned different NDMI cell keys.")

    parts = []
    for day, (mean_column, coverage_column) in bands_by_day.items():
        if coverage_column not in wide.columns:
            raise ValueError(
                f"Earth Engine did not return {coverage_column}. "
                f"Returned columns: {wide.columns.tolist()}"
            )
        coverage = pd.to_numeric(wide[coverage_column])
        mean = pd.to_numeric(wide.get(mean_column, pd.Series(float("nan"), index=wide.index)))
        if not coverage.between(0, 1).all():
            raise ValueError(f"Invalid NDMI coverage in {coverage_column}.")
        if (coverage.gt(0) & mean.isna()).any():
            raise ValueError(f"NDMI coverage is positive without a value for {mean_column}.")
        parts.append(wide[CELL_KEYS].assign(**{
            NDMI_SOURCE_DATE: day,
            "ndmi_mean": mean,
            NDMI_COVERAGE: coverage,
        }))
    return pd.concat(parts, ignore_index=True)


In [ ]:
SMAP_SOURCE_DAY = "smap_observation_day_utc"
NDMI_SOURCE_DATE = "ndmi_composite_start_utc"
NDMI_COVERAGE = "ndmi_valid_fraction"


def load_static(columns, snapshot_file, fallback=True):
    path = FEATURE_DIR / snapshot_file
    required = [*CELL_KEYS, *columns]
    if path.is_file():
        return align_cells(pd.read_parquet(path, columns=required))
    combined = FEATURE_DIR / "environmental_features.parquet"
    if fallback and combined.is_file():
        from pyarrow.parquet import read_schema

        if set(required).issubset(read_schema(combined).names):
            saved = pd.read_parquet(combined, columns=required).groupby(CELL_KEYS, as_index=False)
            if saved.nunique()[columns].gt(1).any(axis=None):
                raise ValueError(f"Use the static input {snapshot_file}; its values vary in the combined table.")
            return align_cells(saved.first())
    return pd.DataFrame(np.nan, columns=columns, index=CELL_INDEX)


def static_features(issue):
    terrain = load_static(TERRAIN, "elevation_by_cell.parquet").apply(numeric)
    terrain["slope_mean_deg"] = terrain["slope_mean_deg"].where(terrain["slope_mean_deg"].between(0, 90))
    land = pd.DataFrame(np.nan, columns=LAND, index=CELL_INDEX)
    for vintage, release in sorted(manifest["corine_release_dates_used"].items(), reverse=True):
        if issue >= pd.Timestamp(release, tz="UTC"):
            land = load_static(LAND, f"corine_{vintage}_by_cell.parquet", fallback=vintage == "2018").apply(numeric)
            if land.notna().any(axis=None):
                break
    land = land.where(land.ge(0) & land.le(1))
    return pd.concat([terrain, land], axis=1)



def history_index(days):
    return pd.MultiIndex.from_product([days, ys, xs], names=KEYS)


def fetch_smap_history(days, regions):
    obs_days = pd.date_range(days[0] - pd.Timedelta(days=14), days[-1] - pd.Timedelta(days=8), freq="D")
    smap = (ee.ImageCollection("NASA/SMAP/SPL4SMGP/008").select("sm_surface")
            .filterDate(obs_days[0].isoformat(), (obs_days[-1] + pd.Timedelta(days=1)).isoformat()))
    dense = pd.DataFrame(np.nan, index=obs_days, columns=CELL_INDEX)
    if smap.size().getInfo():
        projection = ee.Image(smap.first()).projection().getInfo()
        daily = fetch_smap_block(ee, smap, projection, regions, grid_cells, obs_days)
        daily[SMAP_SOURCE_DAY] = utc(daily[SMAP_SOURCE_DAY])
        dense = daily.pivot(index=SMAP_SOURCE_DAY, columns=CELL_KEYS, values="sm_surface").reindex(
            index=obs_days, columns=CELL_INDEX,
        )
        dense = dense.where(dense.ge(0) & dense.le(1))
    source_days = days - pd.Timedelta(days=8)
    index = history_index(days)
    return pd.DataFrame({
        SMAP[0]: dense.loc[source_days].to_numpy().ravel(),
        SMAP[1]: dense.rolling(7).mean().loc[source_days].to_numpy().ravel(),
        SMAP_SOURCE_DAY: source_days.repeat(len(CELL_INDEX)),
    }, index=index)


def fetch_ndmi_history(days, regions):
    delay = pd.Timedelta(days=11)
    fallback = pd.Timedelta(days=24)
    latest, earliest = days[-1] - delay, days[0] - delay - fallback
    modis = (ee.ImageCollection("MODIS/061/MOD09A1")
             .filterDate(earliest.isoformat(), (latest + pd.Timedelta(days=1)).isoformat())
             .select(["sur_refl_b02", "sur_refl_b06", "StateQA", "QA"]))
    dates = pd.DatetimeIndex(pd.to_datetime(
        modis.aggregate_array("system:time_start").getInfo(), unit="ms", utc=True,
    )).as_unit("ns").sort_values()
    idx = history_index(days)
    if dates.empty:
        return pd.DataFrame({"ndmi_mean": np.nan, NDMI_SOURCE_DATE: pd.NaT}, index=idx)
    if not dates.is_unique or not dates.equals(dates.normalize()):
        raise ValueError("Expected unique MODIS composite dates at midnight.")
    projection = ee.Image(modis.first()).select("sur_refl_b02").projection().getInfo()
    table = pd.concat([
        fetch_ndmi_block(ee, modis, projection, regions, grid_cells, dates[i:i + 6])
        for i in range(0, len(dates), 6)
    ], ignore_index=True)
    table[CELL_KEYS] = table[CELL_KEYS].astype(grid_cells.dtypes)
    table[NDMI_SOURCE_DATE] = utc(table[NDMI_SOURCE_DATE])
    table["available_utc"] = table[NDMI_SOURCE_DATE] + delay
    usable = table.loc[table[NDMI_COVERAGE].between(0.20, 1) & table["ndmi_mean"].between(-1, 1)]
    matched = pd.merge_asof(
        idx.to_frame(index=False).sort_values("issue_utc"),
        usable.sort_values("available_utc"),
        left_on="issue_utc", right_on="available_utc", by=CELL_KEYS,
        direction="backward", tolerance=fallback,
    )
    return matched.set_index(KEYS)[["ndmi_mean", NDMI_SOURCE_DATE]].reindex(idx)


## Fire Weather Index

The live calculation uses the same GFS noon-weather recipe and `xclim` equations as training. It starts 120 days before the earliest FWI needed for the issue, so its moisture state can differ slightly from the continuously initialized training archive. Complete weather blocks are cached on Drive for reuse.


In [ ]:
def clean_gfs_values(frame, names):
    temperature = names[0]
    for name in names:
        values = pd.to_numeric(frame[name], errors="raise")
        invalid = np.isinf(values)
        if name == temperature:
            invalid |= values.lt(-273.15)
            negative = pd.Series(False, index=values.index)
        else:
            negative = values.lt(0) & ~invalid

        flagged = invalid | negative
        if flagged.any():
            dates = frame.loc[flagged, "issue_utc"]
            original = values[flagged]
            print(
                f"GFS quality {name} {dates.min():%Y-%m-%d} to {dates.max():%Y-%m-%d}: "
                f"{int(negative.sum())} clipped to zero, {int(invalid.sum())} set to NaN; "
                f"original values {original.min():.6g} to {original.max():.6g}.",
                flush=True,
            )
        frame[name] = values.mask(invalid).mask(negative, 0)
    return frame


def fwi_day_image(ee, archive, issue):
    run_18 = issue - pd.Timedelta(hours=RUN_AGE_HOURS)
    noon = issue + pd.Timedelta(hours=12)

    def select_at(valid, latest_run, source_bands, names):
        candidates = []
        for backup_hours in (24, 6, 0):
            run = latest_run - pd.Timedelta(hours=backup_hours)
            lead = int((valid - run) / pd.Timedelta(hours=1))
            matching = (archive.filter(ee.Filter.eq("creation_time", run.value // 10**6))
                        .filter(ee.Filter.eq("forecast_time", valid.value // 10**6))
                        .filter(ee.Filter.eq("forecast_hours", lead)))
            empty = ee.Image.constant([0] * len(names)).rename(names).selfMask().toFloat()
            forecast = (ee.Image(matching.first()).select(source_bands, names).toFloat()
                        .resample("bilinear").reproject(crs="EPSG:4326", crsTransform=transform))
            forecast = forecast.updateMask(forecast.mask().reduce(ee.Reducer.min()))
            candidates.append(ee.Image(ee.Algorithms.If(matching.size().eq(1), forecast, empty)))
        return ee.ImageCollection.fromImages(candidates).mosaic()

    state = select_at(
        noon, run_18, [*FWI_GFS_BANDS, FWI_GFS_RAIN], ["temperature", "humidity", "u", "v", "rain"],
    )
    wind = state.select("u").hypot(state.select("v")).multiply(3.6)
    rain_bands = [FWI_GFS_RAIN]
    rain_intervals = [
        select_at(run_18, run_18 - pd.Timedelta(hours=6), rain_bands, ["rain"]),
        *[select_at(run_18 + pd.Timedelta(hours=lead), run_18, rain_bands, ["rain"])
          for lead in (6, 12)],
        state.select("rain"),
    ]
    # Adding the bands directly keeps the daily total masked if any interval is missing.
    rain = rain_intervals[0].add(rain_intervals[1]).add(rain_intervals[2]).add(rain_intervals[3])
    bands = state.select(["temperature", "humidity"]).addBands([wind, rain])
    return bands.toFloat().unmask(MISSING_PIXEL).rename([
        f"d{issue:%Y%m%d}_{name}" for name in FWI_WEATHER
    ])


def fetch_fwi_weather(ee, block):
    archive = ee.ImageCollection(GFS_DATASET).filter(ee.Filter.inList(
        "forecast_hours", list(range(6, 43, 6)),
    ))
    stack = ee.Image.cat(*[fwi_day_image(ee, archive, day) for day in block])
    pixels = compute_pixels(stack, "FWI")
    grid_cells = pd.MultiIndex.from_product([ys[::-1], xs], names=KEYS[1:]).to_frame(index=False)
    frames = []
    for day in block:
        frame = grid_cells.assign(issue_utc=day)
        for name in FWI_WEATHER:
            values = pixels[f"d{day:%Y%m%d}_{name}"].astype(np.float32).ravel()
            frame[name] = np.where(values == MISSING_PIXEL, np.nan, values)
        frames.append(frame[[*KEYS, *FWI_WEATHER]])
    return pd.concat(frames, ignore_index=True)


def calculate_fwi(weather, days):
    import xarray as xr
    from xclim.indices.fire import cffwis_indices

    weather = weather.assign(issue_utc=pd.to_datetime(weather["issue_utc"], utc=True))
    weather = clean_gfs_values(weather, FWI_WEATHER).set_index(KEYS, verify_integrity=True)
    grid = pd.MultiIndex.from_product([days, ys, xs], names=KEYS)
    if len(grid.difference(weather.index)) or len(weather.index.difference(grid)):
        raise ValueError("FWI weather does not cover the expected dates and cells.")

    values = weather.reindex(grid)[FWI_WEATHER].to_numpy(dtype=float, copy=True).reshape(len(days), -1, 4)
    # The FWI equations include fractional powers of (100 - RH).
    values[:, :, 1] = np.minimum(values[:, :, 1], 100)
    valid = np.isfinite(values).all(axis=2)
    fwi = np.full(valid.shape, np.nan, dtype=np.float32)
    latitudes = (np.repeat(ys, len(xs)) + 0.5) * CELL_DEG
    dates = days.tz_localize(None)

    # Group cells with the same missing days so each run can use contiguous valid weather.
    _, groups = np.unique(valid.T, axis=0, return_inverse=True)
    for group in np.unique(groups):
        cells = np.flatnonzero(groups == group)
        usable_days = np.flatnonzero(valid[:, cells[0]])
        segments = np.split(usable_days, np.flatnonzero(np.diff(usable_days) > 1) + 1)
        previous_codes = {}
        lat = xr.DataArray(latitudes[cells], dims="cell", attrs={"units": "degrees_north"})
        for segment in segments:
            if not len(segment):
                continue
            data = {
                arg: xr.DataArray(values[segment[:, None], cells, idx], dims=("time", "cell"),
                                  coords={"time": dates[segment]}, attrs={"units": unit})
                for idx, (arg, unit) in enumerate((
                    ("tas", "degC"), ("hurs", "%"), ("sfcWind", "km/h"), ("pr", "mm/day"),
                ))
            }
            indices = cffwis_indices(**data, lat=lat, season_method=None, **previous_codes)
            fwi[np.ix_(segment, cells)] = indices.FWI.transpose("time", "cell").values
            # Missing days retain the last modeled moisture state and have no FWI value.
            previous_codes = {name.lower() + "0": getattr(indices, name).isel(time=-1, drop=True).copy()
                              for name in ("FFMC", "DMC", "DC")}

    return pd.DataFrame({"fwi": fwi.ravel()}, index=grid).reset_index()

FWI_GFS_BANDS = GFS_STATE_BANDS
FWI_GFS_RAIN = GFS_RAIN_BAND
FWI_WEATHER = ["fwi_temp_c", "fwi_rh_pct", "fwi_wind_kmh", "fwi_rain_mm"]


def fetch_fwi_history(history_days):
    need = history_days[-8:]
    days = pd.date_range(need[0] - pd.Timedelta(days=120), need[-1], freq="D")
    signature = sha256(repr((GFS_DATASET, transform, "noon_utc_v1")).encode()).hexdigest()[:12]
    cache_dir = OUTPUT_ROOT / "cache" / f"fwi_{signature}"
    cache_dir.mkdir(parents=True, exist_ok=True)
    parts = []
    # Fixed calendar blocks let forecasts issued on different days reuse the cache.
    cache_anchor = pd.Timestamp("2015-07-02", tz="UTC")
    block_start = days[0] - pd.Timedelta(days=(days[0] - cache_anchor).days % 14)
    for start in pd.date_range(block_start, days[-1], freq="14D"):
        end = min(start + pd.Timedelta(days=13), days[-1])
        block = pd.date_range(start, end, freq="D")
        path = cache_dir / f"{start:%Y%m%d}_{end:%Y%m%d}.parquet"
        if path.is_file():
            part = pd.read_parquet(path)
        else:
            print(f"GFS for FWI · {start:%Y-%m-%d} to {end:%Y-%m-%d}", flush=True)
            part = fetch_fwi_weather(ee, block)
            if part[FWI_WEATHER].notna().all(axis=None):
                part.to_parquet(path, index=False)
        parts.append(part)
    weather = pd.concat(parts, ignore_index=True)
    scores = calculate_fwi(weather.loc[utc(weather["issue_utc"]).isin(days)].copy(), days)
    return scores.set_index(KEYS)[["fwi"]].reindex(history_index(history_days))


## Assemble the saved model's feature columns

This cell applies the training notebook's trailing weather windows, prior FWI mean, changes in SMAP and NDMI, observation ages, and cumulative forecast rain. It checks the feature names and order against the saved model manifest.


In [ ]:
def build_temporal_features(panel, panel_index):
    # Each matrix row is a UTC issue day. Each column is one fixed grid cell.
    if not pd.MultiIndex.from_frame(panel[KEYS]).equals(panel_index):
        raise ValueError("Temporal features require the full, ordered daily grid.")

    n_cells = len(CELL_INDEX)
    values = {}

    def matrix(column):
        return pd.DataFrame(panel[column].to_numpy(dtype=np.float64).reshape(-1, n_cells))

    def add(column, daily_grid):
        values[column] = np.asarray(daily_grid, dtype=np.float32).ravel()

    # Historical weather summaries use Day 1 forecasts from past issues.
    # Shift first so today's target-day weather is outside these windows.
    for column, extreme in zip(GFS[1][:3], ("max", "min", "max")):
        previous = matrix(column).shift(1)
        for window, min_days in ((3, 2), (7, 5)):
            history = previous.rolling(window, min_periods=min_days)
            add(f"{column}_mean_prior_{window}d", history.mean())
            add(f"{column}_{extreme}_prior_{window}d", getattr(history, extreme)())

    previous_rain = matrix(GFS[1][-1]).shift(1)
    for window in (3, 7, 14):
        add(f"gfs_precip_sum_prior_{window}d", previous_rain.rolling(window).sum())

    previous_fwi = matrix("fwi").shift(1)
    add("fwi_mean_prior_7d", previous_fwi.rolling(7, min_periods=5).mean())

    for column, lag in (("smap_surface_daily", 7), ("ndmi_mean", 8)):
        series = matrix(column)
        add(f"{column}_change_{lag}d", series - series.shift(lag))


    for column, date_column in ((SMAP[0], SMAP_SOURCE_DAY), ("ndmi_mean", NDMI_SOURCE_DATE)):
        age = (panel["issue_utc"] - utc(panel[date_column])).dt.days
        add(f"{column}_age_days", age.where(panel[column].notna() & age.ge(0)))

    for lead in HORIZONS[1:]:
        rain = [GFS[day][-1] for day in range(1, lead + 1)]
        add(f"gfs_rain_through_day{lead}", panel[rain].sum(axis=1, min_count=lead))
    return pd.DataFrame(values, index=panel.index, copy=False)


def build_inputs(issue):
    days = pd.date_range(issue - pd.Timedelta(days=14), issue, freq="D")
    idx = history_index(days)
    warnings = []
    weather = fetch_weather(days).set_index(KEYS, verify_integrity=True).reindex(idx)
    static = static_features(issue).reindex(idx.droplevel("issue_utc")).set_axis(idx)
    regions = cell_regions(ee, grid_cells)
    tables = [weather, static]
    for label, load, fallback in (
        ("SMAP", lambda: fetch_smap_history(days, regions), {SMAP[0]: np.nan, SMAP[1]: np.nan, SMAP_SOURCE_DAY: pd.NaT}),
        ("Terra NDMI", lambda: fetch_ndmi_history(days, regions), {"ndmi_mean": np.nan, NDMI_SOURCE_DATE: pd.NaT}),
        ("FWI", lambda: fetch_fwi_history(days), {"fwi": np.nan}),
    ):
        try:
            tables.append(load())
        except ee.EEException as error:
            warnings.append(f"{label} unavailable from Earth Engine.")
            print(f"{label}: {error}")
            tables.append(pd.DataFrame(fallback, index=idx))

    panel = pd.concat(tables, axis=1).reset_index()
    panel["lat_center"] = ((panel["cell_y"] + 0.5) * CELL_DEG).astype(np.float32)
    panel["lon_center"] = ((panel["cell_x"] + 0.5) * CELL_DEG).astype(np.float32)
    phase = 2 * np.pi * (panel["issue_utc"].dt.dayofyear - 1) / 365.25
    panel["season_sin"] = np.sin(phase).astype(np.float32)
    panel["season_cos"] = np.cos(phase).astype(np.float32)
    panel = pd.concat([panel, build_temporal_features(panel, idx)], axis=1)
    missing = sorted(set(all_features).difference(panel.columns))
    if missing:
        raise ValueError(f"Inference is missing training columns: {missing}")
    rows = panel.loc[panel["issue_utc"].eq(issue)].set_index(CELL_KEYS, verify_integrity=True)
    inputs = rows[all_features].apply(numeric).astype(np.float32).reindex(CELL_INDEX)
    return inputs, warnings


## Date-and-location interface and JSON export

The output's `days[i]` corresponds to `series.noaa20_detection_probability[i]`. For the grid output, each row of that matrix follows the order of `cells`. The point output has one probability per day. All dates use UTC. Probabilities are 0–1.


In [ ]:
SERIES = "noaa20_detection_probability"
CELL_META = [
    {
        "cell_y": int(y), "cell_x": int(x),
        "lat_center": float((y + 0.5) * CELL_DEG),
        "lon_center": float((x + 0.5) * CELL_DEG),
        "bbox_wsen": [float(x * CELL_DEG), float(y * CELL_DEG),
                      float((x + 1) * CELL_DEG), float((y + 1) * CELL_DEG)],
    }
    for y, x in CELL_INDEX
]


def score_grid(inputs):
    scores = {}
    for lead in HORIZONS:
        weather_present = inputs[GFS[lead]].notna().any(axis=1)
        probability = pd.Series(np.nan, index=CELL_INDEX, dtype=float)
        if weather_present.any():
            probability.loc[weather_present] = models[lead].predict_proba(
                inputs.loc[weather_present, FEATURES[lead]]
            )[:, 1]
        scores[lead] = [float(value) if pd.notna(value) else None for value in probability]
    return scores


@lru_cache(maxsize=2)
def score_issue(issue):
    inputs, warnings = build_inputs(issue)
    scores = score_grid(inputs)
    empty = [name for name in all_features if inputs[name].isna().all()]
    if empty:
        warnings.append("Some model features are missing across the entire grid. See missing_model_features.")
    return scores, empty, warnings


def forecast_grid(date, days=10):
    if isinstance(days, bool) or not isinstance(days, (int, np.integer)) or days < 1:
        raise ValueError("days must be a positive integer.")
    days = int(days)
    issue = issue_timestamp(date)
    scores, empty, warnings = score_issue(issue)
    leads = HORIZONS[:days]
    dates = [(issue + pd.Timedelta(days=lead - 1)).strftime("%Y-%m-%d") for lead in leads]
    matrix = [scores[lead].copy() for lead in leads]
    scored = [sum(value is not None for value in row) for row in matrix]
    return {
        "id": "spain",
        "name": "Central Spain",
        "kind": "daily_grid_forecast",
        "geometry_note": "Forecast for the trained 0.25 degree grid in central Spain.",
        "bbox_wsen": [WEST, SOUTH, EAST, NORTH],
        "cell_degrees": CELL_DEG,
        "cells": CELL_META,
        "days": dates,
        "series": {SERIES: matrix},
        "quality": {
            "unit": "probability (0–1)",
            "target": "At least one NOAA-20 FIRMS detection in the cell during the UTC day",
            "model_run": RUN_DIR.name,
            "scored_cells_by_day": scored,
            "missing_model_features": empty.copy(),
        },
        "data_policy": {
            "issue_utc": issue.isoformat(),
            "gfs_run_utc": (issue - pd.Timedelta(hours=RUN_AGE_HOURS)).isoformat(),
            "requested_days": days,
            "trained_max_days": max(HORIZONS),
        },
        "warnings": [
            *warnings,
            *([f"The saved models support up to {max(HORIZONS)} days per issue."] if days > max(HORIZONS) else []),
            *(["A null score means no GFS fields for that cell and horizon were available."]
              if any(count < len(CELL_INDEX) for count in scored) else []),
        ],
    }


def predict(date, latitude, longitude, days=10):
    latitude, longitude = float(latitude), float(longitude)
    if not (SOUTH <= latitude < NORTH and WEST <= longitude < EAST):
        raise ValueError(f"Location outside model grid: longitude [{WEST}, {EAST}), latitude [{SOUTH}, {NORTH}).")
    index = CELL_INDEX.get_loc((int(np.floor(latitude / CELL_DEG)), int(np.floor(longitude / CELL_DEG))))
    grid = forecast_grid(date, days)
    probabilities = [row[index] for row in grid["series"][SERIES]]
    quality = {**grid["quality"], "available_by_day": [value is not None for value in probabilities]}
    del quality["scored_cells_by_day"]
    return {
        "id": grid["id"],
        "name": grid["name"],
        "kind": "daily_point_forecast",
        "geometry_note": "Forecast for the grid cell containing the requested coordinates.",
        "location": {"latitude": latitude, "longitude": longitude, **CELL_META[index]},
        "days": grid["days"],
        "series": {SERIES: probabilities},
        "quality": quality,
        "data_policy": grid["data_policy"],
        "warnings": grid["warnings"],
    }


def save_forecast(forecast, history_path=None, output_path=None):
    if history_path is None:
        history_path = next((path for path in (
            DRIVE_ROOT / "spain.json", DRIVE_ROOT.parent / "spain.json", Path("/content/spain.json"),
        ) if path.is_file()), None)
    if output_path is None:
        issue_date = forecast["data_policy"]["issue_utc"][:10]
        output_path = OUTPUT_ROOT / issue_date / "spain_forecast.json"
    output_path = Path(output_path)
    payload = forecast
    if history_path is not None:
        history_path = Path(history_path)
        if output_path.resolve() == history_path.resolve():
            raise ValueError("Choose a separate output file to preserve the original spain.json.")
        history = json.loads(history_path.read_text(encoding="utf-8"))
        if history.get("id") != "spain" or "weeks" not in history:
            raise ValueError("HISTORY_JSON must be the weekly Spain file.")
        payload = {**history, "forecast": forecast}
    output_path.parent.mkdir(parents=True, exist_ok=True)
    output_path.write_text(json.dumps(payload, ensure_ascii=False, allow_nan=False) + "\n", encoding="utf-8")
    return output_path


In [ ]:
forecast = forecast_grid(ISSUE_DATE, days=REQUESTED_DAYS)
output = save_forecast(forecast, HISTORY_JSON, OUTPUT_PATH)
print(f"Saved {output}")
display(pd.DataFrame({
    "date_utc": forecast["days"],
    "scored_cells": forecast["quality"]["scored_cells_by_day"],
    "trained_cells": len(forecast["cells"]),
}))

# Point example:
predict("2026-11-15", latitude=40.4168, longitude=-3.7038, days=10)
